# 🔁 GIADA Task 30 — voltaggio autonomo, singolo compartimento
Calcio e stimoli restano imposti. La mappa di voltaggio semi-implicita è identica nei bracci formula e ibrido; confronti teacher-voltage e autoregressivi separano errore diretto e feedback. Il controllo nativo NEURON riguarda soltanto il solver passivo, non l'intero compartimento attivo.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,uuid
from IPython.display import display,Javascript
WORK=Path('/kaggle/working')/('giada_task30_'+uuid.uuid4().hex[:8]);WORK.mkdir()
REPO=WORK/'giada'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',os.environ.get('GIADA_CODE_REVISION','codex/surrogate-validity-audit')],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
try:
 import neuron
except ImportError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+')[0]=='8.2.7',neuron.__version__
import torch
assert torch.cuda.is_available(),'Task30 richiede CUDA'
print({'revision':REVISION,'neuron':neuron.__version__,'gpu':torch.cuda.get_device_name()})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_task30_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
completed=subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_task30_autonomous_voltage.py'),'--output',str(OUTPUT)],check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').exists() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'error':f'worker interrupted: {completed.returncode}'}
display({k:report.get(k) for k in ('valid','scientific_primary_passed','gate_d_performance_status','task31_authorized','autonomous_voltage','full_active_neuron_native_validated','error')})
if report.get('family_primary'):display(report['family_primary'])
print({'output':str(OUTPUT),'exit':completed.returncode})


In [ ]:
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',OUTPUT))
if os.environ.get('GIADA_MANUAL_DOWNLOAD','0')=='1':
 payload=base64.b64encode(archive.read_bytes()).decode('ascii')
 display(Javascript(f"const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})
